In [ ]:
from importlib import reload
from pathlib import Path

import data_labeling_pipeline.rhe
from data_labeling_pipeline.grandtour_missions import (
    get_mission_by_index,
)

import data_labeling_pipeline
from data_labeling_pipeline import DataLabelingPipeline

reload(data_labeling_pipeline)

reload(data_labeling_pipeline.rhe)


# Currently downloaded
downloaded_missions = [0, 16, 18, 30]
mission_index = 30
print(f"Mission index: {mission_index}")
mission = get_mission_by_index(mission_index)
# mission = "2024-11-04-16-52-38"

mission_path = Path("/datahome/lmertens/grand_tour_dataset") / mission
assert mission_path.exists(), f"Mission path {mission_path} does not exist"

output_dir = Path("./output") / mission
# output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
pipeline = DataLabelingPipeline(mission_path=mission_path)


pipeline.process_mission(output_dir=output_dir)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from data_labeling_pipeline.dataset_loader import GrandTourLoader

# reload(GrandTourLoader)

loader = GrandTourLoader(mission_path)
shared_ts, U, Z = loader.load_data_shared_timestamps(dt=0.1, fill_gaps=True)

v_cmd, omega_cmd = U.T
px, py, yaw = Z.T
vx = np.gradient(px, shared_ts)
vy = np.gradient(py, shared_ts)
v_forward = np.cos(yaw) * vx + np.sin(yaw) * vy
omega_act = np.gradient(np.unwrap(yaw), shared_ts)

dv = np.abs(v_cmd - v_forward)
d_omega = np.abs(omega_cmd - omega_act)

# Load traversability labels and align to shared timeline
labels_dir = Path(f"./output/{Path(mission_path).name}")
label_ts = np.load(labels_dir / "timestamp.npy")
trav_labels = np.load(labels_dir / "trav_labels.npy")
nu_confidences = np.load(labels_dir / "nu_confidences.npy")

mu_label = trav_labels[:, 0]
nu_label = trav_labels[:, 1]
delta_theta_label = trav_labels[:, 2]


mu = np.interp(shared_ts, label_ts, mu_label)
nu = np.interp(shared_ts, label_ts, nu_label)
delta_theta = np.interp(shared_ts, label_ts, delta_theta_label)
nu_confidence = np.interp(shared_ts, label_ts, nu_confidences)

fig, axes = plt.subplots(7, 1, figsize=(13, 13), sharex=True)

axes[0].plot(shared_ts, v_cmd, label="v_cmd")
axes[0].plot(shared_ts, v_forward, label="v_forward (body x)", alpha=0.85)
axes[0].set_ylabel("Linear [m/s]")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(shared_ts, dv, label="dv = abs(v_cmd - v_forward)")
axes[1].plot(shared_ts, 1 - mu, label="1-mu", color="tab:green")
axes[1].set_ylabel("Error vs. traversability label (linear)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(shared_ts, omega_cmd, label="omega_cmd")
axes[2].plot(shared_ts, omega_act, label="omega_act = d(yaw)/dt", alpha=0.5)
axes[2].set_ylabel("Angular [rad/s]")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

axes[3].plot(shared_ts, d_omega, label="d_omega = abs(omega_cmd - omega_act)")
axes[3].plot(shared_ts, 1 - nu, label="1-nu", color="tab:orange")
axes[3].set_ylabel("Error vs. traversability label (angular)")
axes[3].set_xlabel("Time [s]")
axes[3].legend()
axes[3].grid(True, alpha=0.3)

axes[4].plot(shared_ts, nu_confidence, label="nu_confidence", color="tab:purple")
axes[4].set_ylabel("Angular Confidence")
axes[4].set_xlabel("Time [s]")
axes[4].legend()
axes[4].grid(True, alpha=0.3)

axes[5].plot(shared_ts, delta_theta, label="delta_theta")
axes[5].set_ylabel("Heading error label [rad]")
axes[5].set_xlabel("Time [s]")
axes[5].legend()
axes[5].grid(True, alpha=0.3)

# axes[5].plot(shared_ts, yaw, label="yaw")
# axes[5].set_ylabel("Yaw [rad]")
# axes[5].set_xlabel("Time [s]")
# axes[5].legend()

# Plot the pearson correcelation with a rolling window
window_size = shared_ts.shape[0] - 1
window_size = 100


rolling_corr_v = np.array([
    np.corrcoef(dv[i : i + window_size], 1 - mu[i : i + window_size])[0, 1]
    for i in range(len(dv) - window_size)
])
rolling_corr_omega = np.array([
    np.corrcoef(d_omega[i : i + window_size], 1 - nu[i : i + window_size])[0, 1]
    for i in range(len(d_omega) - window_size)
])

print(rolling_corr_v)
print(rolling_corr_omega)
axes[6].plot(
    shared_ts[: len(rolling_corr_v)], rolling_corr_v, label="Rolling Corr (dv vs 1-mu)"
)
axes[6].plot(
    shared_ts[: len(rolling_corr_omega)],
    rolling_corr_omega,
    label="Rolling Corr (d_omega vs 1-nu)",
)
axes[6].set_ylabel("Rolling Pearson Correlation")
axes[6].set_xlabel("Time [s]")
axes[6].legend()
axes[6].grid(True, alpha=0.3)


# Add red shading where commands are near zero
def add_zero_shading(ax, time, condition, color="red", alpha=0.2):
    diff = np.diff(condition.astype(int))
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0]
    if condition[0]:
        starts = np.concatenate([[0], starts])
    if condition[-1]:
        ends = np.concatenate([ends, [len(condition) - 1]])
    for s, e in zip(starts, ends):
        ax.axvspan(
            time[s],
            time[e],
            color=color,
            alpha=alpha,
        )


add_zero_shading(axes[1], shared_ts, np.isclose(v_cmd, 0))
add_zero_shading(axes[3], shared_ts, np.isclose(omega_cmd, 0))
# add_zero_shading(axes[3], shared_ts, np.abs(omega_cmd) > 0.001, color="green")

fig.tight_layout()
plt.show()
